In [1]:
from train import train

model_variant = "resnet50" #50, 101, droneranger

model, metrics = train(
    data_root="/mnt/active_storage/Knut/LRDD_v3",
    metadata_dir="/mnt/active_storage/Knut/LRDD_v3/metadata",
    backbone=model_variant,
    crop_only=True, # single backbone drone crop for training when true, otherwise uses double backbone crop + full image
    use_huber=True, # use huber loss instead of MSE loss
    use_droneranger=False, # only set this to true when using dronerange backbone
    use_alt_head=True, # uses alternate prediction head, otherwise defaults to dronerange prediction head
    epochs=50,
    batch_size=32,
    lr=1e-3,
    patience=50,
    checkpoint_path=model_variant+"_huber_alt",
    metrics_path=model_variant+"_huber_alt_training_metrics.json",
    num_workers=32,
)

#checkpoint_weights="resnet50_huber_latest.pth"


[scan_dataset] !! 01-24-2025/IMG_0001: no matching CSV (strict) → skip
[scan_dataset] !! 02-12-2025/PXL_0002: no matching CSV (strict) → skip
[LRDDDataset] WARN: 1 rows dropped because distance_3d_ft was missing/invalid in 04-11-2025_DJI_0004_metadata.csv
[LRDDDataset] INFO: kept 1073 / 1074 rows for 04-11-2025_DJI_0004_metadata.csv
[LRDDDataset] WARN: 1 rows dropped because image file was missing in /mnt/active_storage/Knut/LRDD_v3/train/04-18-2025/DJI_0075/images
[LRDDDataset] INFO: kept 257 / 258 rows for 04-18-2025_DJI_0075_metadata.csv
[LRDDDataset] WARN: 1 rows dropped because image file was missing in /mnt/active_storage/Knut/LRDD_v3/train/04-18-2025/DJI_0078/images
[LRDDDataset] INFO: kept 401 / 402 rows for 04-18-2025_DJI_0078_metadata.csv
[LRDDDataset] WARN: 2 rows dropped because image file was missing in /mnt/active_storage/Knut/LRDD_v3/train/04-18-2025/DJI_0085/images
[LRDDDataset] INFO: kept 1634 / 1636 rows for 04-18-2025_DJI_0085_metadata.csv
[LRDDDataset] WARN: 1 rows 

OutOfMemoryError: CUDA out of memory. Tried to allocate 256.00 MiB. GPU 0 has a total capacity of 23.68 GiB of which 234.12 MiB is free. Including non-PyTorch memory, this process has 23.32 GiB memory in use. Of the allocated memory 22.99 GiB is allocated by PyTorch, and 38.30 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
%matplotlib inline
import json
import matplotlib.pyplot as plt

with open("resnet50_huber_alt_training_metrics.json") as f:
    metrics = json.load(f)

plt.plot(metrics["epoch"], metrics["train_loss"], label="Train Loss")
plt.plot(metrics["epoch"], metrics["val_loss"], label="Val Loss")
plt.plot(metrics["epoch"], metrics["val_mae"], label="Val MAE")
plt.plot(metrics["epoch"], metrics["val_rmse"], label="Val RMSE")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from evaluate import evaluate_model

model_variant = "resnet50" #"droneranger"#

evaluate_model(
    data_root="/mnt/active_storage/Knut/LRDD_v3",
    metadata_dir="/mnt/active_storage/Knut/LRDD_v3/metadata",
    checkpoint_path=model_variant+"_huber_alt_best.pth",
    backbone=model_variant,
    use_droneranger=False,
    use_alt_head=True,
    batch_size=16,
    device="cuda",
    save_predictions=False,
    num_workers=32,
    max_dist=10000 # set max distance in ft to include for testing. Use 10000 to include all data. I've also been running for 400, 300, and 200ft
)